## Task to solve
**Business Task:** 
Build an ML model that predicts future occupancy/demand for each listing and gives hosts recommendations on how to increase bookings

**Business Problem**
Airbnb hosts and property managers want to maximize their revenue and occupancy but struggle to understand which factors truly drive booking success and high guest satisfaction. Many listings underperform despite good locations, while others outperform expectations.
Goal: Build an ML system that predicts the future success of a listing and provides actionable recommendations to hosts on how to improve performance.

**Target variable** = Occupancy Rate next 30 days

**Key Features You Can Use**
**Host-related:**

host_response_time, host_response_rate, host_acceptance_rate
host_is_superhost, host_identity_verified
calculated_host_listings_count (multi-listing hosts vs single)
host_since (experience)

**Property & Listing:**

property_type, room_type
accommodates, bedrooms, bathrooms_text
amenities (needs parsing)
instant_bookable

**Location & Market:**

neighbourhood_cleansed, latitude, longitude
neighbourhood_group_cleansed (if available in other cities)

**Listing Quality:**

Length and sentiment of description + neighborhood_overview
name length/sentiment (NLP features)

**Availability & History:**

availability_30/60/90/365
minimum_nights, maximum_nights

## Different ways
**Market Segmentation / Clustering**

* Task: Group similar listings into segments (budget, luxury, family, business, unique experiences, etc.).
* Type: Unsupervised (K-Means, DBSCAN, Hierarchical, or embedding-based clustering).
* Features: Property type, amenities (parsed), location, price behavior, occupancy patterns.
* Value: Targeted marketing, personalized recommendations, competitive analysis.

**Superhost / High-Performer Prediction**

* Task: Predict which hosts/listings will achieve Superhost status or high long-term success.
* Target: host_is_superhost or a custom "High Performer" label (based on reviews + occupancy).
* Type: Classification (or Ranking).
* Value: Help new hosts understand what changes they need to reach top status.

**Listing Churn / Deactivation Prediction**

* Task: Predict which listings are likely to be removed or become inactive in the next 3-6 months.
* Target: Create label from availability table (e.g., "inactive if no bookings in 60+ days").
* Type: Binary classification or Survival Analysis.
* Value: Platform can intervene early with support for at-risk hosts.

## Data preparation

### Listing

#### Грузим данные

In [1]:
import pandas as pd

listing_path = "initial_data/listings.csv.gz"
listing_df = pd.read_csv(listing_path)

In [9]:
listing_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 81853 entries, 0 to 81852
Data columns (total 79 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            81853 non-null  int64  
 1   listing_url                                   81853 non-null  str    
 2   scrape_id                                     81853 non-null  int64  
 3   last_scraped                                  81853 non-null  str    
 4   source                                        81853 non-null  str    
 5   name                                          81853 non-null  str    
 6   description                                   79140 non-null  str    
 7   neighborhood_overview                         39600 non-null  str    
 8   picture_url                                   81851 non-null  str    
 9   host_id                                       81853 non-null  int64  
 1

#### Удаляем ненужные столбцы

In [178]:
listing_df2 = listing_df.copy()

# Определяем список колонок к удалению
drop_list = []
for col in listing_df.columns:

    # Добавляем в список полностью пустые колонки
    if listing_df[col].isnull().mean() == 1:
        drop_list += [col]
        continue

    # Добавляем в список колонки с одним значением
    if listing_df[col].nunique() == 1:
        drop_list += [col]
        continue

    # Добавляем в список колонки с ссылками
    if 'url' in col:
        drop_list += [col]
        continue

# Дополнительно убираем поля
drop_list += ['host_name',  # Имя хоста. Бесполензо для модели
              'estimated_occupancy_l365d',  # Cлишком сильно может коррелировать с нашей метрикой
              'host_id',  # id хоста
              'availability_60',  # То же самое, что и таргет в 30 дн. Сильно коррелирует с таргетом
                'availability_90',  # То же самое, что и таргет в 30 дн. Сильно коррелирует с таргетом
                'availability_365',  # То же самое, что и таргет в 30 дн. Сильно коррелирует с таргетом
                'availability_eoy',  # То же самое, что и таргет в 30 дн. Сильно коррелирует с таргетом
                'id',  # id объекта
                'calendar_last_scraped',  # дата скржпинга
            ]

listing_df2.drop(columns=drop_list, inplace=True)

# Предобразуем таргет
listing_df2['availability_30'] = listing_df2['availability_30'] / 30

#### Кодируем время ответа

In [179]:
# Кодируем время ответа
responce_time_coding = {
    'within an hour': 1,
    'within a few hours': 3,
    'within a day': 24,
    'a few days or more': 72,
}
listing_df2['host_response_time'] = listing_df2['host_response_time'].replace(responce_time_coding)

# Заполняем незаполнные значения максимальной категорией
listing_df2['host_response_time'] = listing_df2['host_response_time'].fillna(72)  


#### Обратабывает отвечаемость и подтверждаемость хоста

In [180]:

for col in ['host_response_rate', 'host_acceptance_rate']:
    # Переводим из % в float
    listing_df2[col] = listing_df2[col].str.replace('%', '').apply(float) / 100

    # Заполняем пропуски 0
    listing_df2[col] = listing_df2[col].fillna(0)


#### Способы верификации хоста

In [181]:
import ast 

# Преобразуем строку в реальный список py
listing_df2['host_verifications'] = listing_df2['host_verifications'].apply(lambda x: ast.literal_eval(x) if not pd.isna(x) else None)

# Раскрываем список, делаем энкодинг и схлопываем до изначального размера
dummies_df = pd.get_dummies(listing_df2['host_verifications'].explode(), prefix='host_verifications').groupby(level=0).max()

# Объединяем с исходным df
listing_df2 = pd.concat([listing_df2, dummies_df], axis=1)


#### Длительность на платформе

In [182]:
# Преобразуем в даты
listing_df2['host_since'] = pd.to_datetime(listing_df2['host_since'])
listing_df2['last_scraped'] = pd.to_datetime(listing_df2['last_scraped'])

# Считаем возраст хоста на платформе
listing_df2['host_age_on_platform'] = (listing_df2['last_scraped'] - listing_df2['host_since']).dt.days / 365

listing_df2['host_age_on_platform'] = listing_df2['host_age_on_platform'].fillna(0)

# Дропаем ненужное
listing_df2.drop(columns=['host_since', 'last_scraped'], inplace=True)

#### Форматируем булевые колонки

In [183]:
bool_columns_list = [
    'host_is_superhost',
    'host_has_profile_pic',
    'host_identity_verified',
    'instant_bookable',
]

for col in bool_columns_list:

    listing_df2[col] = listing_df2[col].replace({'f': False, 't': True})

    listing_df2[col] = listing_df2[col].fillna(False)


### Считаем таргет

In [184]:
# # calendar_path = "initial_data/calendar.csv.gz"
# calendar_df = pd.read_csv(calendar_path)

In [ ]:
# calendar_df['date'] = pd.to_datetime(calendar_df['date'])

In [ ]:
# from datetime import timedelta

# target_df = pd.merge(
#     left=listing_df2[['id', 'calendar_last_scraped']],
#     right=calendar_df,
#     left_on='id',
#     right_on='listing_id',
#     how='left',
# )

# target_df['calendar_last_scraped'] = pd.to_datetime(target_df['calendar_last_scraped'])

# # Определяем правую границу интервала дат
# target_df['date_to'] = target_df['calendar_last_scraped'] + timedelta(days=30)

# # Оставлем тлк данные за  первые 30 дней после даты скрэпинга и только занаятые дни
# target_df = target_df[
#         (target_df['date'] >= target_df['calendar_last_scraped']) &
#         (target_df['date'] < target_df['date_to']) &
#         (target_df['available'] == 'f')  
#     ]

# # Считаем кол-во дней заняости в ближайшие 30 дней
# target_df = target_df.groupby(by='id', as_index=False)['date'].nunique()

# # Считаем занятость в ближайшие 30 дней
# target_df['date'] = target_df['date'] / 30

# # Переименовываем
# target_df.rename(columns={'date': 'occupancy_30d'}, inplace=True)

#### Джоиним к основной витрине с фичами

In [ ]:
# listing_df3 = pd.merge(
#     left=listing_df2,
#     right=target_df,
#     on='id',
#     how='left',
# )

# # Если ничего не заджоинилось, значит объект полностью свободен
# listing_df3['occupancy_30d'] = listing_df3['occupancy_30d'].fillna(0)

In [187]:
listing_df2.to_pickle(r'D:\AV\Education\Master\ML\Project\prepared_data\data.pkl')

## Regression

In [79]:
import pandas as pd


df = pd.read_pickle(r"C:\Users\alrvt\Downloads\data (2).pkl")
df.head()

,name,description,neighborhood_overview,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_neighbourhood,...,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,host_verifications_email,host_verifications_phone,host_verifications_work_email,host_age_on_platform
0,Nice flat close to Montparnasse,This beautiful apartment is very well located ...,"The neighborhood is very nice, feel free to sh...","Paris, France","I am living in Paris since 10 years, and I lov...",72,0.0,0.00,False,Invalides - Ecole Militaire,...,False,1,1,0,0,0.57,True,True,False,11.446575
1,"Marais - Charming loft, river view","In the heart of historical Paris le Marais, gr...",Le Marais is the heart of Paris. Few years ago...,"Paris, France","I travel a lot for work and when I'm away, my ...",3,1.0,0.92,True,Saint-Paul - Ile Saint-Louis,...,False,1,1,0,0,0.66,True,True,False,11.446575
2,Superbe 2 pièces idéalement placé,A stone's throw from Parc Monceau and the Arc ...,Quiet and residential area... Close to the Bat...,"Paris, France",NaN,72,0.0,0.00,False,Monceau,...,False,1,1,0,0,NaN,True,True,False,11.471233
3,"Cozy place, Sacré Coeur Montmartre in the SoPi",Cozy apartment close to everything.<br />Perfe...,One of the most trendy corners of the capital ...,"Paris, France","Jeune ingénieur, je propose mon appartement à ...",1,1.0,0.89,False,Pigalle - Saint-Georges,...,False,1,1,0,0,0.72,True,True,False,12.172603
4,Appartement d'architecte - Montmartre,"Our apartment, refurbished by an architect in ...",The apartment is in the heart of Paris's 18th ...,"Paris, France",NaN,1,1.0,1.00,False,Vaugirard,...,False,1,1,0,0,0.25,True,True,False,11.443836


In [82]:
df2 = df.copy()

df2['host_response_time'] = df2['host_response_time'].apply(int)

for col in ['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified', 'instant_bookable']:
    df2[col] = df2[col].apply(bool)

drop_col_list = df2.select_dtypes(exclude=['int', 'bool', 'float']).columns

df2 = df2.drop(columns=list(drop_col_list) + ['latitude', 'longitude'])

# Для признаков, содержащих мало пропусков, просто выкидываю значения
few_null_values_col_list = [
    'host_listings_count',
    'host_total_listings_count',                         
    'minimum_minimum_nights',     
    'maximum_minimum_nights',       
    'minimum_maximum_nights',
    'maximum_maximum_nights',
]
for col in few_null_values_col_list:
    df2 = df2[~df2[col].isna()]

# Признаки, содержащие много пропусков пока просто удаляю
many_null_values_col_list = [
    'review_scores_rating',
    'review_scores_accuracy',
    'review_scores_cleanliness',
    'review_scores_checkin',
    'review_scores_communication',
    'review_scores_location',
    'review_scores_value',
    'reviews_per_month',
    'bedrooms',
]

df2 = df2.drop(columns=many_null_values_col_list)

print('Кол-во пропусков = ', df2.isnull().sum().sum())
print(df2.shape)

Кол-во пропусков =  0
(81820, 31)


### KNN

In [88]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error

In [84]:
X = df2.drop(columns=['availability_30'])
y = df2['availability_30']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [94]:
from tqdm import tqdm

In [99]:


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) 
X_test_scaled = scaler.transform(X_test)

res = {
    'n': [],
    'rmse': [],
}

for n in tqdm(range(2, 40)):
    knn = KNeighborsRegressor(
        n_neighbors=n
    )

    knn.fit(X_train_scaled, y_train)

    y_pred = knn.predict(X_test_scaled)

    res['n'] += [n]
    res['rmse'] += [root_mean_squared_error(
        y_true=y_test,
        y_pred=y_pred,
    )]

pd.DataFrame(res).plot(kind='line', x='n', y='rmse')

100%|██████████| 38/38 [00:26<00:00,  1.44it/s]


ImportError: matplotlib is required for plotting when the default backend "matplotlib" is selected.

In [77]:
X_train.isna().sum().sum()

np.int64(14424)

In [ ]:
# # Обратабываем локацию хоста. Если не известно проставлем False, полагая, что это так же не приклекает
# df2['host_paris_location'] = df2['host_location'].apply(lambda x: 'Paris' in x if not pd.isna(x) else False)


# property_type
# room_type

# bathrooms_text

# amenities

# first_review
# last_review

# license

,name,description,neighborhood_overview,host_location,host_about,host_neighbourhood,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,property_type,room_type,bathrooms_text,amenities,first_review,last_review,license,instant_bookable
0,Nice flat close to Montparnasse,This beautiful apartment is very well located ...,"The neighborhood is very nice, feel free to sh...","Paris, France","I am living in Paris since 10 years, and I lov...",Invalides - Ecole Militaire,"[email, phone]",True,True,Palais-Bourbon,Entire rental unit,Entire home/apt,1 bath,"[""Wifi"", ""Hot water"", ""Heating"", ""Coffee maker...",2014-05-17,2020-03-13,7510702406383,False
1,"Marais - Charming loft, river view","In the heart of historical Paris le Marais, gr...",Le Marais is the heart of Paris. Few years ago...,"Paris, France","I travel a lot for work and when I'm away, my ...",Saint-Paul - Ile Saint-Louis,"[email, phone]",True,True,Hôtel-de-Ville,Entire rental unit,Entire home/apt,1 bath,"[""Paid parking on premises"", ""Wifi"", ""Free was...",2014-10-29,2025-08-29,7510400455677,False
2,Superbe 2 pièces idéalement placé,A stone's throw from Parc Monceau and the Arc ...,Quiet and residential area... Close to the Bat...,"Paris, France",NaN,Monceau,"[email, phone]",True,False,Batignolles-Monceau,Entire rental unit,Entire home/apt,1 bath,"[""Hair dryer"", ""Iron"", ""TV"", ""Heating"", ""Wifi""...",NaN,NaN,NaN,False
3,"Cozy place, Sacré Coeur Montmartre in the SoPi",Cozy apartment close to everything.<br />Perfe...,One of the most trendy corners of the capital ...,"Paris, France","Jeune ingénieur, je propose mon appartement à ...",Pigalle - Saint-Georges,"[email, phone]",True,True,Opéra,Entire rental unit,Entire home/apt,1 bath,"[""Elevator"", ""Hot water kettle"", ""Wifi"", ""Laun...",2014-05-25,2025-07-03,7510901014972,False
4,Appartement d'architecte - Montmartre,"Our apartment, refurbished by an architect in ...",The apartment is in the heart of Paris's 18th ...,"Paris, France",NaN,Vaugirard,"[email, phone]",True,True,Buttes-Montmartre,Entire rental unit,Entire home/apt,1 bath,"[""Elevator"", ""Wifi"", ""Self check-in"", ""Hot wat...",2023-05-21,2025-07-20,7511808333770,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81848,Apt a/c 1BR/2P - Saint-Lazare/Les Champs Elysée,📌 Checkmyguest offers you a charming and spaci...,🏘️ The 8th arrondissement of Paris is one of t...,"Paris, France",Checkmyguest est une entreprise de gestion loc...,NaN,"[email, phone]",True,True,Élysée,Entire rental unit,Entire home/apt,1 bath,"[""Elevator"", ""Hot water kettle"", ""Wifi"", ""Dini...",2025-08-22,2025-09-14,7510815621419,True
81849,Superbe duplex avec terrasse face Tour Eiffel,The Konciergerie is proud to present this char...,NaN,"Paris, France","Bienvenue à Paris ! \nJe suis Eliav, passionné...",NaN,"[email, phone]",True,True,Vaugirard,Entire rental unit,Entire home/apt,1.5 baths,"[""Hot water kettle"", ""Wifi"", ""Babysitter recom...",NaN,NaN,"Available with a mobility lease only (""bail mo...",True
81850,Cozy studio - 2P - Place des Vosges/Marais,Checkmyguest offers you an exceptional 24 m² s...,"Romantic, festive and trendy, the Marais distr...",NaN,NaN,NaN,"[email, phone]",True,True,Hôtel-de-Ville,Entire rental unit,Entire home/apt,1 bath,"[""Hot water kettle"", ""Wifi"", ""Paid street park...",2025-09-07,2025-09-07,7510406440280,True
81851,Appt entier 35m2 proche Montmartre,Quiet 35 m2 apartment in the heart of Paris.<b...,NaN,"Paris, France","Mon appartement parisien est très cosy, décoré...",Pigalle - Saint-Georges,"[email, phone]",True,True,Opéra,Private room in rental unit,Private room,1 private bath,"[""Wifi"", ""Laundromat nearby"", ""Freezer"", ""Self...",NaN,NaN,NaN,False


In [45]:
for c in df2.dtypes:
    print(c)

str
str
str
str
str
int64
float64
float64
bool
str
float64
float64
object
object
object
str
float64
float64
str
str
int64
str
float64
str
int64
int64
float64
float64
float64
float64
float64
float64
float64
int64
int64
int64
int64
str
str
float64
float64
float64
float64
float64
float64
float64
str
object
int64
int64
int64
int64
float64
bool
bool
bool
float64
bool
